In [48]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# Create the initial model
model_initial = NeuralNetwork().to(device)

# Save its initial weights and biases
initial_state = model_initial.state_dict()

# Create two models
model_sgd = NeuralNetwork().to(device)
model_adam = NeuralNetwork().to(device)

# Give both models the exact same starting parameters
model_sgd.load_state_dict(initial_state)
model_adam.load_state_dict(initial_state)

Using device: cuda


<All keys matched successfully>

In [49]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):

        X = X.to(device)
        y = y.to(device)

        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [50]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_sgd.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model_sgd, loss_fn, optimizer)
    test_loop(test_dataloader, model_sgd, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.292453  [   64/60000]
loss: 2.287192  [ 6464/60000]
loss: 2.267178  [12864/60000]
loss: 2.269224  [19264/60000]
loss: 2.247752  [25664/60000]
loss: 2.214337  [32064/60000]
loss: 2.222849  [38464/60000]
loss: 2.191204  [44864/60000]
loss: 2.173548  [51264/60000]
loss: 2.151804  [57664/60000]
Test Error: 
 Accuracy: 51.4%, Avg loss: 2.147420 

Epoch 2
-------------------------------
loss: 2.149584  [   64/60000]
loss: 2.148005  [ 6464/60000]
loss: 2.091783  [12864/60000]
loss: 2.115649  [19264/60000]
loss: 2.062694  [25664/60000]
loss: 1.994651  [32064/60000]
loss: 2.028620  [38464/60000]
loss: 1.954749  [44864/60000]
loss: 1.944719  [51264/60000]
loss: 1.881804  [57664/60000]
Test Error: 
 Accuracy: 58.3%, Avg loss: 1.885213 

Epoch 3
-------------------------------
loss: 1.914984  [   64/60000]
loss: 1.888045  [ 6464/60000]
loss: 1.781372  [12864/60000]
loss: 1.824241  [19264/60000]
loss: 1.709320  [25664/60000]
loss: 1.657723  [32064/600

In [51]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_adam.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model_adam, loss_fn, optimizer)
    test_loop(test_dataloader, model_adam, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.292453  [   64/60000]
loss: 0.548226  [ 6464/60000]
loss: 0.405508  [12864/60000]
loss: 0.507536  [19264/60000]
loss: 0.464675  [25664/60000]
loss: 0.434799  [32064/60000]
loss: 0.379426  [38464/60000]
loss: 0.526008  [44864/60000]
loss: 0.461065  [51264/60000]
loss: 0.495713  [57664/60000]
Test Error: 
 Accuracy: 84.3%, Avg loss: 0.427951 

Epoch 2
-------------------------------
loss: 0.255355  [   64/60000]
loss: 0.355397  [ 6464/60000]
loss: 0.297590  [12864/60000]
loss: 0.386969  [19264/60000]
loss: 0.436076  [25664/60000]
loss: 0.365925  [32064/60000]
loss: 0.321627  [38464/60000]
loss: 0.487858  [44864/60000]
loss: 0.395104  [51264/60000]
loss: 0.444074  [57664/60000]
Test Error: 
 Accuracy: 85.5%, Avg loss: 0.392907 

Epoch 3
-------------------------------
loss: 0.204157  [   64/60000]
loss: 0.339616  [ 6464/60000]
loss: 0.237889  [12864/60000]
loss: 0.315665  [19264/60000]
loss: 0.380235  [25664/60000]
loss: 0.331275  [32064/600